# Label robustness: LayerZero's initial Sybil list (A16)

The labels are LayerZero's bounty-accepted list (`fcfs_list.csv`). By the bounty rules, that list excludes
LayerZero's **initial list** (LayerZero, Nansen and Chaos Labs, including self-reported addresses), so
initial-list addresses among the interactors are labeled non-Sybil. This notebook measures how many there are and
how much they move the reported metrics. The list and its provenance are in `review_support/initial_list/`.

1. **Counts**: interactors on the initial list, by category and by partition (seed-42 tree split).
2. **Baseline**: LightGBM, 3 seeds, parameters from `02`, trained here (same machine as every variant).
3. **False-positive composition**: test false positives that are on the initial list, at the validation threshold
   and the fixed operating points.
4. **Variant A, clean negatives**: initial-list negatives removed from training and validation (same partition
   indices), training re-upsampled to 1:1 as in `sp.make_splits`, retrained, re-thresholded on the cleaned
   validation set. 2×2 table: model × test set (full test, test without initial-list rows).
5. **Variant B, relabel as positive (exploratory)**: `y = sybil | initial list`, same partitions. The target
   changes to "any LayerZero-identified Sybil".
6. **10 splits**: baseline and Variant A on `sp.SPLIT_SEEDS` (model seed 42), paired differences.

Thresholds come from validation only. **Results**: `results/16_label_robustness_initial_list.json`.

## Setup

In [1]:
import hashlib, os, platform, time, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.utils import resample
import sybil_pipeline as sp
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
assert not sp.CODE_COMMIT.endswith('-dirty'), f'Run from a clean working tree (CODE_COMMIT={sp.CODE_COMMIT})'
print('Code commit:', sp.CODE_COMMIT)
PLATFORM = dict(machine=platform.machine(), processor=platform.processor(), system=platform.system(),
                python=platform.python_version(), cpu_count=os.cpu_count(), n_jobs=sp.N_JOBS)
print('Platform:', PLATFORM)

DATA_DIR = './data'
FEATS = sp.FEATS
LIST_PATH = 'review_support/initial_list/initial_list.csv'
LIST_SHA256 = '1a643f70056fe0c0b38d4a13468e968a2de0bd18de6bc25ed7139d9fa691a7cc'
sha = hashlib.sha256(open(LIST_PATH, 'rb').read()).hexdigest()
assert sha == LIST_SHA256, f'{LIST_PATH}: SHA-256 {sha} differs from the recorded value (run git lfs pull?)'
lst = pd.read_csv(LIST_PATH, dtype=str)['ADDRESS'].str.strip().str.lower()
assert lst.is_unique
initial_set = set(lst)
print(f'Initial list: {len(lst):,} addresses ({lst.str.fullmatch(r"0x[0-9a-f]{40}").sum():,} 20-byte, '
      f'{lst.str.fullmatch(r"0x[0-9a-f]{64}").sum():,} 32-byte)')

df, _, _ = sp.build_master_df(DATA_DIR, verbose=False)
init = df['addr'].isin(initial_set)
syb = df['sybil'] == 1
_, LGBM_SELECTED = sp.load_selected_params()

Code commit: 5888aa3
Platform: {'machine': 'arm64', 'processor': 'arm', 'system': 'Darwin', 'python': '3.13.7', 'cpu_count': 10, 'n_jobs': 4}


Initial list: 803,093 addresses (801,932 20-byte, 1,161 32-byte)


## 1. Counts

In [2]:
fcfs = pd.read_csv(sp.data_paths(DATA_DIR)['sybil'])
fcfs_set = set(fcfs.iloc[:, 0].str.lower())
S = sp.make_splits(df, FEATS, method='group', seed=sp.SEED)
part = pd.Series('Train', index=df.index)
part.loc[S['idx_val']], part.loc[S['idx_test']] = 'Validation', 'Test'
counts = dict(
    list_size=len(initial_set), overlap_with_fcfs_list=len(initial_set & fcfs_set),
    interactors=len(df), interactors_on_list=int(init.sum()),
    on_list_and_sybil=int((init & syb).sum()),
    negatives=int((~syb).sum()), negatives_on_list=int((init & ~syb).sum()),
    share_of_negatives=float((init & ~syb).sum() / (~syb).sum()),
)
assert counts['on_list_and_sybil'] == 0
by_cat = (pd.DataFrame({'category': df['category'], 'init': init, 'neg': ~syb})
          .groupby('category').agg(addresses=('init', 'size'), on_list=('init', 'sum'))
          .assign(share=lambda t: t['on_list'] / t['addresses']))
by_part = (pd.DataFrame({'partition': part, 'init': init & ~syb, 'neg': ~syb})
           .groupby('partition').agg(negatives=('neg', 'sum'), negatives_on_list=('init', 'sum'))
           .assign(share=lambda t: t['negatives_on_list'] / t['negatives']).loc[['Train', 'Validation', 'Test']])
for k, v in counts.items():
    print(f'  {k:<25} {v:,.4f}' if isinstance(v, float) else f'  {k:<25} {v:,}')
print(by_cat.round(4).to_string()); print(by_part.round(4).to_string())

  list_size                 803,093
  overlap_with_fcfs_list    0
  interactors               434,786
  interactors_on_list       34,545
  on_list_and_sybil         0
  negatives                 416,575
  negatives_on_list         34,545
  share_of_negatives        0.0829
             addresses  on_list   share
category                               
IxE              93061     2669  0.0287
IxI              34391     2206  0.0641
IxL             306658    29646  0.0967
No provider        676       24  0.0355
            negatives  negatives_on_list   share
partition                                       
Train          204122              17145  0.0840
Validation      87481               7085  0.0810
Test           124972              10315  0.0825


## 2. Baseline and the split helper

`with_partitions` rebuilds a split from fixed partition indices with optional rows dropped and optional new labels,
and re-upsamples the training partition to 1:1 exactly as `sp.make_splits` does (`resample(replace=True,
random_state=seed)`, then a shuffle with the same seed). Without changes it reproduces `sp.make_splits`.

In [3]:
def with_partitions(S, df, y, seed, drop=None):
    """Split dict on S's partition indices, labels y, rows in `drop` removed from train and validation."""
    drop = pd.Series(False, index=df.index) if drop is None else drop
    tr = S['idx_train'][~drop.loc[S['idx_train']].to_numpy()]
    va = S['idx_val'][~drop.loc[S['idx_val']].to_numpy()]
    te = S['idx_test']
    X = df[FEATS].astype(np.float32)
    trn = pd.concat([X.loc[tr], y.loc[tr].rename('sybil')], axis=1)
    maj, mn = trn[trn['sybil'] == 0], trn[trn['sybil'] == 1]
    mn_up = resample(mn, replace=True, n_samples=len(maj), random_state=seed)
    bal = pd.concat([maj, mn_up]).sample(frac=1, random_state=seed)
    return dict(method=S['method'], X_train=bal.drop(columns='sybil').astype(np.float32), y_train=bal['sybil'],
                X_val=X.loc[va], y_val=y.loc[va], X_test=X.loc[te], y_test=y.loc[te],
                idx_train=tr, idx_val=va, idx_test=te)

y0 = df['sybil'].astype(int)
S_chk = with_partitions(S, df, y0, sp.SEED)
assert S_chk['X_train'].equals(S['X_train']) and S_chk['y_train'].equals(S['y_train'])   # reproduces make_splits
del S_chk

def metrics(y, p, thr):
    y, yhat = np.asarray(y), (p >= thr).astype(int)
    return dict(n=len(y), sybils=int(y.sum()), threshold=thr, precision=precision_score(y, yhat, zero_division=0),
                recall=recall_score(y, yhat, zero_division=0), f1=f1_score(y, yhat, zero_division=0),
                auroc=roc_auc_score(y, p), ap=average_precision_score(y, p), flagged=int(yhat.sum()))

t0 = time.time()
base = sp.fit_lgbm(S, LGBM_SELECTED)
thr_base = sp.best_f1_threshold(S['y_val'], base['val'])
res_base = sp.evaluate('LightGBM (tuned, 3-seed), this machine', S['y_val'], base['val'], S['y_test'], base['test'])
assert res_base['threshold'] == thr_base
sp.print_metrics(res_base)
print(f'({time.time() - t0:.0f}s)')

  LightGBM seed=42  rounds=1224  39.6s


  LightGBM seed=123  rounds=1267  41.8s


  LightGBM seed=456  rounds=1225  42.8s


  Threshold : 0.5748
  Precision : 0.7149
  Recall    : 0.7187
  F1        : 0.7168
  AUROC     : 0.9683
  Avg Prec  : 0.7697
  Brier     : 0.0195
(124s)


## 3. False-positive composition (baseline, test)

In [4]:
te_init = init.loc[S['idx_test']].to_numpy()
y_te = S['y_test'].to_numpy()
neg = y_te == 0
assert int((neg & te_init).sum()) + int((neg & ~te_init).sum()) == int(neg.sum())   # counts reconcile
rows = []
for t in sorted({0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.50, thr_base}, reverse=True):
    flag = base['test'] >= t
    tp, fp = int((flag & ~neg).sum()), int((flag & neg).sum())
    fp_init = int((flag & neg & te_init).sum())
    rows.append(dict(threshold=round(t, 4), note='validation threshold' if t == thr_base else '', flagged=int(flag.sum()),
                     tp=tp, fp=fp, fp_on_initial_list=fp_init, fp_not_on_list=fp - fp_init,
                     fp_share_on_list=fp_init / fp if fp else np.nan,
                     precision=tp / (tp + fp), precision_if_list_counted_sybil=(tp + fp_init) / (tp + fp)))
fp_comp = pd.DataFrame(rows)
print(f"Test negatives: {int(neg.sum()):,}, of which on the initial list {int((neg & te_init).sum()):,} "
      f"({(neg & te_init).sum() / neg.sum():.2%})")
print(fp_comp.round(4).to_string(index=False))

Test negatives: 124,972, of which on the initial list 10,315 (8.25%)
 threshold                 note  flagged   tp   fp  fp_on_initial_list  fp_not_on_list  fp_share_on_list  precision  precision_if_list_counted_sybil
    0.9500                          2991 2618  373                  65             308            0.1743     0.8753                           0.8970
    0.9000                          3656 3071  585                 118             467            0.2017     0.8400                           0.8723
    0.8500                          4058 3325  733                 178             555            0.2428     0.8194                           0.8632
    0.8000                          4362 3489  873                 236             637            0.2703     0.7999                           0.8540
    0.7500                          4647 3617 1030                 331             699            0.3214     0.7784                           0.8496
    0.7000                          4

## 4. Variant A: clean negatives

Initial-list negatives are removed from training and validation; the test partition is unchanged. Both models are
scored on the full test set and on the test set without initial-list rows, each at its own validation threshold.

In [5]:
t0 = time.time()
S_A = with_partitions(S, df, y0, sp.SEED, drop=init & ~syb)
assert (S_A['idx_test'] == S['idx_test']).all()                       # same test rows
print(f"Variant A: train {len(S_A['idx_train']):,} unique rows (was {len(S['idx_train']):,}), "
      f"validation {len(S_A['idx_val']):,} (was {len(S['idx_val']):,}); upsampled train {len(S_A['X_train']):,}")
mA = sp.fit_lgbm(S_A, LGBM_SELECTED)
thr_A = sp.best_f1_threshold(S_A['y_val'], mA['val'])
print(f'({time.time() - t0:.0f}s)')

clean = ~te_init
table = []
for model, p, thr in [('Baseline', base['test'], thr_base), ('Variant A (clean negatives)', mA['test'], thr_A)]:
    for test_set, k in [('full test', np.ones_like(clean)), ('test without initial-list rows', clean)]:
        table.append(dict(model=model, test_set=test_set, **metrics(y_te[k], p[k], thr)))
table = pd.DataFrame(table)
print(table.round(4).to_string(index=False))

Variant A: train 195,901 unique rows (was 213,046), validation 84,220 (was 91,305); upsampled train 373,954


  LightGBM seed=42  rounds=961  35.0s


  LightGBM seed=123  rounds=990  31.2s


  LightGBM seed=456  rounds=952  32.1s


(98s)
                      model                       test_set      n  sybils  threshold  precision  recall     f1  auroc     ap  flagged
                   Baseline                      full test 130435    5463     0.5748     0.7149  0.7187 0.7168 0.9683 0.7697     5492
                   Baseline test without initial-list rows 120120    5463     0.5748     0.8067  0.7187 0.7601 0.9721 0.8069     4867
Variant A (clean negatives)                      full test 130435    5463     0.5078     0.5518  0.7862 0.6484 0.9597 0.4569     7784
Variant A (clean negatives) test without initial-list rows 120120    5463     0.5078     0.7872  0.7862 0.7867 0.9742 0.8322     5456


## 5. Variant B: initial-list addresses relabeled as Sybil (exploratory)

The target becomes "any LayerZero-identified Sybil" (bounty list or initial list), on the same partitions. Not
comparable with the other rows as a detection result; it shows how the task changes under the wider label.

In [6]:
t0 = time.time()
yB = (syb | init).astype(int)
S_B = with_partitions(S, df, yB, sp.SEED)
assert (S_B['idx_test'] == S['idx_test']).all()
mB = sp.fit_lgbm(S_B, LGBM_SELECTED)
rB = sp.evaluate('LightGBM, Variant B labels', S_B['y_val'], mB['val'], S_B['y_test'], mB['test'])
variant_B = dict(prevalence_all=float(yB.mean()), prevalence_test=float(S_B['y_test'].mean()),
                 test_positives=int(S_B['y_test'].sum()),
                 **{k: float(rB[k]) for k in ['threshold', 'precision', 'recall', 'f1', 'auroc', 'ap']},
                 recall_on_fcfs_sybils=float((mB['test'][y_te == 1] >= rB['threshold']).mean()),
                 recall_on_initial_list=float((mB['test'][te_init] >= rB['threshold']).mean()))
print(f'({time.time() - t0:.0f}s)')
print({k: round(v, 4) for k, v in variant_B.items()})

  LightGBM seed=42  rounds=2308  66.0s


  LightGBM seed=123  rounds=2218  67.3s


  LightGBM seed=456  rounds=2174  57.5s


(191s)
{'prevalence_all': 0.1213, 'prevalence_test': 0.121, 'test_positives': 15778, 'threshold': 0.4292, 'precision': 0.7863, 'recall': 0.7057, 'f1': 0.7438, 'auroc': 0.9222, 'ap': 0.798, 'recall_on_fcfs_sybils': 0.8135, 'recall_on_initial_list': 0.6487}


## 6. 10 splits: baseline vs Variant A (model seed 42)

In [7]:
rows = []
for s in sp.SPLIT_SEEDS:
    t0 = time.time()
    Ss = sp.make_splits(df, FEATS, method='group', seed=s)
    Sa = with_partitions(Ss, df, y0, s, drop=init & ~syb)
    assert (Sa['idx_test'] == Ss['idx_test']).all()
    yt = Ss['y_test'].to_numpy(); cl = ~init.loc[Ss['idx_test']].to_numpy()
    row = dict(split_seed=s)
    for name, SS in [('base', Ss), ('A', Sa)]:
        r = sp.fit_lgbm(SS, LGBM_SELECTED, seeds=(42,), verbose=False)
        thr = sp.best_f1_threshold(SS['y_val'], r['val'])
        for tag, k in [('full', np.ones_like(cl)), ('clean', cl)]:
            m = metrics(yt[k], r['test'][k], thr)
            row.update({f'{name}_{tag}_{x}': m[x] for x in ['precision', 'recall', 'f1', 'ap', 'auroc']})
        if s == sp.SPLIT_SEEDS[0] and name == 'base':
            ref = sp.lgbm_fit_eval(Ss, FEATS, LGBM_SELECTED)
            assert abs(ref['test_f1'] - row['base_full_f1']) < 1e-12        # same as sp.lgbm_fit_eval
    rows.append(row)
    print(f"split {s}: F1 full base {row['base_full_f1']:.4f} A {row['A_full_f1']:.4f}; "
          f"clean base {row['base_clean_f1']:.4f} A {row['A_clean_f1']:.4f} ({time.time() - t0:.0f}s)", flush=True)
per_split = pd.DataFrame(rows)
assert len(per_split) == 10 and per_split['split_seed'].is_unique
for tag in ['full', 'clean']:
    for x in ['precision', 'recall', 'f1', 'ap', 'auroc']:
        per_split[f'delta_{tag}_{x}'] = per_split[f'A_{tag}_{x}'] - per_split[f'base_{tag}_{x}']
num = [c for c in per_split.columns if c != 'split_seed']
summary = {f'{c}_{a}': float(getattr(per_split[c], a)()) for c in num for a in ('mean', 'std')}
for tag in ['full', 'clean']:
    summary[f'delta_{tag}_f1_higher_n'] = int((per_split[f'delta_{tag}_f1'] > 0).sum())
print('\nVariant A − baseline (mean ± SD over 10 splits, same machine):')
for tag in ['full', 'clean']:
    for x in ['precision', 'recall', 'f1', 'ap', 'auroc']:
        print(f"  {tag:<5} {x:<9} {summary[f'delta_{tag}_{x}_mean']:+.4f} ± {summary[f'delta_{tag}_{x}_std']:.4f}")

split 42: F1 full base 0.7179 A 0.6430; clean base 0.7542 A 0.7834 (125s)


split 1: F1 full base 0.6908 A 0.6332; clean base 0.7132 A 0.7699 (78s)


split 2: F1 full base 0.7135 A 0.6427; clean base 0.7596 A 0.7822 (85s)


split 3: F1 full base 0.7170 A 0.6422; clean base 0.7496 A 0.7781 (72s)


split 4: F1 full base 0.7168 A 0.6375; clean base 0.7492 A 0.7852 (74s)


...

split 6: F1 full base 0.7079 A 0.6420; clean base 0.7361 A 0.7766 (73s)


split 7: F1 full base 0.7209 A 0.6525; clean base 0.7551 A 0.7886 (86s)


split 8: F1 full base 0.7157 A 0.6527; clean base 0.7466 A 0.7858 (76s)


split 9: F1 full base 0.7154 A 0.6466; clean base 0.7532 A 0.7818 (80s)



Variant A − baseline (mean ± SD over 10 splits, same machine):
  full  precision -0.1774 ± 0.0209
  full  recall    +0.0750 ± 0.0207
  full  f1        -0.0694 ± 0.0063
  full  ap        -0.3103 ± 0.0047
  full  auroc     -0.0094 ± 0.0007
  clean precision -0.0137 ± 0.0151
  clean recall    +0.0750 ± 0.0207
  clean f1        +0.0343 ± 0.0096
  clean ap        +0.0281 ± 0.0019
  clean auroc     +0.0015 ± 0.0007


## Save results

In [8]:
sp.save_results([res_base], '16_label_robustness_initial_list', leakage=sp.leakage_report(df, S, verbose=False), extra=dict(
    initial_list=dict(path=LIST_PATH, sha256=LIST_SHA256, rows=len(initial_set),
                      provenance='review_support/initial_list/README.md'),
    counts=counts, by_category=by_cat.reset_index().to_dict('records'), by_partition=by_part.reset_index().to_dict('records'),
    fp_composition=fp_comp.to_dict('records'), table_2x2=table.to_dict('records'),
    variant_A=dict(threshold=thr_A, rounds=mA['rounds'], train_unique=len(S_A['idx_train']), val=len(S_A['idx_val'])),
    variant_B=variant_B, variant_B_note='exploratory: the target becomes any LayerZero-identified Sybil',
    per_split=per_split.to_dict('records'), summary=summary,
    params=LGBM_SELECTED, platform=PLATFORM))

Saved results/16_label_robustness_initial_list.json
